# 14. How Sequencing Works and How to Judge Its Quality

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `quality.py`

Quality at each position along the reads of a FASTQ file, as a box plot.


In [ ]:
# Quality at each position along the reads of a FASTQ file, as a box plot.
import gzip
import sys

import matplotlib
from Bio import SeqIO

matplotlib.use("svg")
import matplotlib.pyplot as plt

path, picture = sys.argv[1], sys.argv[2]

by_position = []  # by_position[i] holds every quality seen at position i
reads = 0
with gzip.open(path, "rt") as handle:
    for record in SeqIO.parse(handle, "fastq"):
        reads += 1
        for i, q in enumerate(record.letter_annotations["phred_quality"]):
            if i == len(by_position):
                by_position.append([])
            by_position[i].append(q)

means = [sum(qs) / len(qs) for qs in by_position]
print(f"{reads:,} reads, up to {len(by_position)} bases long")
for start in (0, 49, 99, 129, 139, 149):
    if start < len(means):
        mean = means[start]
        print(f"  mean quality at position {start + 1:3}: {mean:5.1f}")

fig, ax = plt.subplots(figsize=(5.6, 2.8))
ax.boxplot(
    by_position[::5],
    positions=range(1, len(by_position) + 1, 5),
    widths=3,
    showfliers=False,
    patch_artist=True,
    boxprops={"facecolor": "#BBBBBB", "color": "#444444"},
    medianprops={"color": "#222222"},
)
ax.axhline(20, color="#777777", linestyle="--", linewidth=0.8)
ax.set_xticks(range(0, len(by_position) + 1, 25))
ax.set_xticklabels(range(0, len(by_position) + 1, 25))
ax.set_ylim(0, 42)
ax.set_xlabel("Position in read")
ax.set_ylabel("Phred quality")
fig.savefig(picture, bbox_inches="tight")
print(f"saved {picture}")

## `adapters.py`

How many reads contain each of the two common Illumina adapters?


In [ ]:
# How many reads contain each of the two common Illumina adapters?
import gzip

from Bio import SeqIO

ADAPTERS = {
    "TruSeq": "AGATCGGAAGAGC",
    "Nextera": "CTGTCTCTTATACACATCT",
}

counts = dict.fromkeys(ADAPTERS, 0)
reads = 0
with gzip.open("reads.fastq.gz", "rt") as handle:
    for record in SeqIO.parse(handle, "fastq"):
        reads += 1
        sequence = str(record.seq)
        for name, adapter in ADAPTERS.items():
            counts[name] += adapter in sequence

for name, count in counts.items():
    print(f"{name:8} {count:6,} of {reads:,} reads ({count / reads:.1%})")

## `trim.py`

Remove adapter and low quality ends from reads, and drop reads left too short to be useful.


In [ ]:
# Remove adapter and low quality ends from reads, and drop reads left too
# short to be useful.
import gzip

from Bio import SeqIO

ADAPTER = "CTGTCTCTTATACACATCT"  # Nextera, the kit this run used
WINDOW, MIN_QUALITY = 4, 20  # trim where 4 bases average below 20
MIN_LENGTH = 50


def trim(record):
    """The record cut at its adapter and its poor quality tail."""
    cut = str(record.seq).find(ADAPTER)
    if cut != -1:
        record = record[:cut]
    qualities = record.letter_annotations["phred_quality"]
    end = len(qualities)
    for start in range(len(qualities) - WINDOW + 1):
        window = qualities[start : start + WINDOW]
        if sum(window) / WINDOW < MIN_QUALITY:
            end = start
            break
    return record[:end]


kept = dropped = adapters = bases_in = bases_out = 0
with (
    gzip.open("reads.fastq.gz", "rt") as source,
    gzip.open("trimmed.fastq.gz", "wt") as target,
):
    for record in SeqIO.parse(source, "fastq"):
        bases_in += len(record)
        adapters += ADAPTER in record.seq
        trimmed = trim(record)
        if len(trimmed) >= MIN_LENGTH:
            SeqIO.write(trimmed, target, "fastq")
            kept += 1
            bases_out += len(trimmed)
        else:
            dropped += 1

print(f"reads with adapter     {adapters:7,}")
print(f"reads kept             {kept:7,}")
print(f"reads dropped          {dropped:7,}")
print(f"bases kept             {bases_out / bases_in:7.1%}")